# RAG Agent — the project in one notebook

This notebook walks through the essence of `01-rag-agent`: a question-answering
agent whose only window into a bounded Wikipedia corpus is two retrieval tools
(`search_documents` / `read_excerpt`), so every answer is grounded in citable
evidence. It demonstrates, in order:

1. **The prepared corpus** — what `dvc repro` materialized into `data/`.
2. **The headline comparison** — the retrieval run ladder persisted in
   `reports/metrics.json` (model vs. seeded random floor, ablations, cost).
3. **One key result, reproduced** — a live retrieval query against the
   persisted FAISS index on a graded question, with its random-ranking floor.

Everything reads from persisted artifacts; nothing is re-downloaded.

## How to run

From the project directory (`projects/01-rag-agent`):

```bash
uv sync --all-packages
```

then run the cells top to bottom (e.g. `jupyter nbconvert --to notebook --execute
notebooks/exploration.ipynb`, or interactively in VS Code/Jupyter).

All data comes from `dvc repro`: the corpus contract and FAISS indexes under
`data/`, and evaluation results under `reports/`. Embedding models load from
the local Hugging Face cache (no network needed). No credentials are required —
the provider fallback chain (ADR-0004) runs everything against the deterministic
mock model when `LLM_BASE_URL`/`LLM_API_KEY` are unset.

## 1. The prepared corpus

The corpus is a bounded, checksum-pinned slice of the English Wikipedia dump
`20231101.en`: five topics, at most 120 articles each plus exact-title anchors,
truncated at 12,000 characters per article. Load it through the package's
loader — the same path the pipeline and evaluation use.

In [ ]:
import json
from collections import Counter

from rag_agent.cli import load_config
from rag_agent.corpus import load_corpus

# RAGConfig defaults overlaid with params.yaml — the same entry point the CLI uses.
config = load_config()
articles = load_corpus(config)

contract = json.loads(config.contract_path.read_text(encoding="utf-8"))
by_topic = Counter(a.topic for a in articles)
word_counts = sorted(len(a.text.split()) for a in articles)

print(f"dataset : {contract['name']} (license: {contract['license']})")
print(f"rows    : {contract['n_rows']}  sha256: {contract['checksum_sha256'][:16]}…")
print(f"topics  : {dict(by_topic)}")
print(
    f"words   : min={word_counts[0]}  "
    f"median={word_counts[len(word_counts) // 2]}  max={word_counts[-1]}"
)

## 2. The headline comparison (from `reports/metrics.json`)

Quality is measured on a graded set of 26 handcrafted questions with known
relevant articles: recall@k and MRR@5 over a bounded grid — three chunk
budgets, an embedding-model arm, a rerank on/off ablation, and a **seeded
uniform-random baseline** marking the floor any ranker must beat. The numbers
below are exactly what the DVC evaluation stage persisted — read, not recomputed.

In [ ]:
from pathlib import Path

metrics = json.loads(Path("reports/metrics.json").read_text(encoding="utf-8"))

ladder = [
    ("chunks=200, MiniLM, rerank ON (chosen)", "retrieval_chunks_200"),
    ("chunks=400, rerank ON", "retrieval_chunks_400"),
    ("chunks=800, rerank ON", "retrieval_chunks_800"),
    ("chunks=200, BGE-small (vs MiniLM)", "retrieval_model_bge_small_chunks_200"),
    ("chunks=200, rerank OFF", "retrieval_rerank_off_chunks_200"),
    ("random baseline (seeded)", "retrieval_random_chunks_200"),
]
print(f"{'configuration':42s} {'recall@5':>9s} {'recall@10':>10s} {'MRR@5':>7s}")
for label, key in ladder:
    m = metrics[key]
    print(f"{label:42s} {m['recall@5']:9.3f} {m['recall@10']:10.3f} {m['mrr@5']:7.3f}")

model = metrics["retrieval_chunks_200"]
floor = metrics["retrieval_random_chunks_200"]
delta = model["recall@5"] - floor["recall@5"]
print(f"\nembeddings beat the random floor by +{delta:.3f} recall@5")
print(f"chosen production config: {metrics['detail']['hpo']['chosen']}")

cost = json.loads(Path("reports/cost_telemetry.json").read_text(encoding="utf-8"))
print(
    f"cost telemetry (mode={cost['mode']}): "
    f"${cost['cost_per_query_usd']['mean']:.2e}/query mean, "
    f"p95 ${cost['cost_per_query_usd']['p95']:.2e}"
)
rag = metrics["rag_agent"]
print(
    f"rag agent (mock mode): faithfulness={rag['faithfulness']:.3f}  "
    f"context_recall={rag['context_recall']:.3f}  p95={rag['p95_latency_s']:.3f}s"
)

## 3. One key result, reproduced: a live retrieval query

The headline claim is that retrieval — not luck — grounds the agent. Reproduce
it on a single graded question: load the persisted FAISS index for the chosen
chunk budget (`data/index/chunks_200`), embed the question with the cached
MiniLM model, rerank with the cross-encoder, and check where the first
relevant article lands. Then compare with the seeded random floor for the same
question — the exact baseline the sweep reports.

In [ ]:
from rag_agent.evaluate import first_relevant_rank, load_questions, random_ranks
from rag_agent.index import (
    CrossEncoderReranker,
    Retriever,
    SentenceTransformerEmbedder,
    load_index,
)

questions = load_questions(config, articles)
q = questions[0]
print(f"question: {q.question}")
print(f"relevant article ids: {q.relevant_article_ids}\n")

best = config.retrieval_best
retriever = Retriever.from_index_dir(
    config,
    config.index_path(best.chunk_size),
    SentenceTransformerEmbedder(best.embedding_model),
    CrossEncoderReranker(config.reranker_model) if best.rerank else None,
)
citations = retriever.search(q.question, top_k=best.top_k)
for i, c in enumerate(citations, start=1):
    print(f"#{i} {c.title} ({c.article_id})")
    print(f"   {c.excerpt}\n")

rank = first_relevant_rank(citations, q.relevant_article_ids)
print(f"first relevant rank (top_k={best.top_k}): {rank}")

In [ ]:
# The same question against the seeded uniform-random floor.
_, chunks = load_index(config.index_path(best.chunk_size))
floor_rank = random_ranks(config, [q], chunks)[0]
print(f"random floor rank for this question: {floor_rank} (over {len(chunks)} chunks)")
print(
    f"embedding retrieval put a relevant passage at #{rank}; "
    f"random ranking put the first one at #{floor_rank}."
)

## Where to go next

- **Case study** — the full narrative, SLOs and business framing:
  [/projects/01-rag-agent](/projects/01-rag-agent)
- **Design record** — chunking guarantees, provider fallback (ADR-0004),
  evaluation harness: [DESIGN.md](../DESIGN.md)
- **Code** — the package behind every cell above: `src/rag_agent/`
  (`corpus.py`, `index.py`, `evaluate.py`, `agent.py`)

Re-run the whole pipeline (corpus → indexes → metrics) with `dvc repro` from
the project directory; the artifacts this notebook read are its outputs.